Hello **Everyone** !  
Welcome to the **second part of Day 2** of our AI pool !  

This morning, you learned how to **fine-tune** a model : you modified GPT-2's weights so it would "memorize" new facts (like false capitals). That works, but it has limits.

This afternoon, we take a completely different approach. Instead of modifying the model, we will **give it access to external documents** and teach it to find the right information before answering.

Our goal : build a system that can answer questions about **your own documents** (PDFs, texts, etc.) with accurate, sourced information.  

By the end of this session, you will have built a complete **RAG** system (Retrieval Augmented Generation) - one of the most widely used techniques in production AI applications today.

**But wait... why not just fine-tune again ?**

Remember this morning ? We fine-tuned GPT-2 to give us false capitals. The model "learned" these false facts by modifying its internal weights.  
But there are real problems with this approach :
- **What if the information changes ?** You would need to re-train every time.
- **What if you have thousands of documents** that update every day ? Fine-tuning is too slow and expensive for that.
- **Fine-tuning is permanent** : once the model learns something wrong, it's hard to "un-learn" it.

We need a smarter approach : **RAG** (Retrieval Augmented Generation).  

Think of it like this :
- **Fine-tuning** = Teaching a student new facts by heart (slow, expensive, hard to update)
- **RAG** = Giving the student access to a library and teaching them how to search (fast, flexible, always up-to-date)

With RAG, the model itself doesn't change. Instead, we **search for relevant information** in our documents and give it to the model along with the question. The model then uses that information to generate an accurate answer.

But before building RAG, we need to understand **embeddings** - the technology that makes intelligent search possible !

# **I/ Understanding Embeddings**

### **What is an Embedding ?**

An embedding is a way to represent text (or images, audio...) as a **list of numbers** (a vector).  

Imagine you want to organize books in a library. Instead of organizing them alphabetically, you organize them by **meaning** :
- Books about cooking are close together
- Books about space are close together
- A book about "cooking in space" would be somewhere in between

Embeddings do exactly this : texts with similar meanings have similar numbers (vectors that are "close" in space).

**Example :**
- "I love pizza" → [0.2, 0.8, 0.1, ...]
- "Pizza is my favorite food" → [0.21, 0.79, 0.12, ...] (very similar)
- "The weather is nice" → [0.9, 0.1, 0.7, ...] (very different)

### ***1/ Setup: Install the necessary packages***

In [ ]:
%pip install sentence-transformers chromadb numpy

### ***2/ Create your first embedding***

We will use a pre-trained model from HuggingFace to create embeddings.  
The model `all-MiniLM-L6-v2` is small, fast, and works great for most use cases.

**Wait, another library ?** This morning we used `transformers` from HuggingFace to load GPT-2 (a text generation model). Now we use `sentence-transformers`, which is built on top of `transformers` but specialized for creating **embeddings**. Think of it this way :
- `transformers` = general-purpose (text generation, classification, translation...)
- `sentence-transformers` = specialized for turning text into vectors (embeddings)

They both come from HuggingFace, but have different purposes.

**Documentation :** https://www.sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html

In [ ]:
from sentence_transformers import SentenceTransformer

# TODO: Load the embedding model 'all-MiniLM-L6-v2'
embedding_model = ...

text = "I love artificial intelligence"

# TODO: Create the embedding
embedding = ...

print(f"Embedding created.")
print(f"Embedding dimension: {len(embedding)}")
print(f"First 10 values: {embedding[:10]}")

### ***3/ Measure similarity between texts***

Now comes the magic. We can measure how **similar** two texts are by comparing their embeddings.  
We use **cosine similarity** : a score between -1 and 1.

**How to read the score :**
- **1.0** = Identical meaning (the two texts say the same thing)
- **0.7 - 0.9** = Very similar (related topics, similar ideas)
- **0.3 - 0.7** = Somewhat related
- **0.0** = No relation at all
- **-1.0** = Opposite meaning

**Intuition :** Imagine each embedding as an arrow in space. Cosine similarity measures the **angle** between two arrows. If they point in the same direction (angle = 0), the similarity is 1. If they point in opposite directions (angle = 180), it's -1.

**Documentation :** https://www.sbert.net/docs/package_reference/util.html

**Your task :** Complete the code to calculate similarity between sentences.

In [ ]:
from sentence_transformers import util

sentences = [
    "I love programming in Python",
    "Python is my favorite programming language",
    "The weather is beautiful today",
    "I enjoy coding and building software"
]

# TODO: Create embeddings for all sentences
embeddings = ...

print("Similarity with 'I love programming in Python':\n")

for i, sentence in enumerate(sentences):
    # TODO: Calculate cosine similarity between first embedding and current one
    similarity = ...
    print(f"  \"{sentence}\"")
    print(f"  → Similarity: {similarity:.4f}\n")

**Question :** Which sentences have the highest similarity ? Does it make sense to you ?  
Take a moment to analyze the results before continuing.

### ***4/ Visualize embeddings in 2D***

We said that embeddings place similar texts "close together" in space. Let's actually **see** this !

The embeddings we created have 384 dimensions - impossible to visualize directly. But we can use a technique called **PCA** (Principal Component Analysis) to compress them down to just 2 dimensions, so we can plot them on a graph.

This is a great way to build intuition about how embeddings organize text by meaning.

**Documentation :** https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

In [ ]:
%pip install matplotlib scikit-learn

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

texts = [
    # Tech topic
    "I love programming in Python",
    "JavaScript is great for web development",
    "Machine learning is fascinating",
    # Food topic  
    "Pizza is my favorite food",
    "I love cooking Italian pasta",
    "Sushi is delicious",
    # Nature topic
    "The mountains are beautiful",
    "I love hiking in the forest",
    "The ocean is peaceful"
]

# TODO: Create embeddings for all texts
text_embeddings = ...

# TODO: Reduce to 2D using PCA
pca = ...
embeddings_2d = ...

plt.figure(figsize=(12, 8))
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red', 'green', 'green', 'green']

for i, (x, y) in enumerate(embeddings_2d):
    plt.scatter(x, y, c=colors[i], s=100)
    plt.annotate(texts[i][:30] + "...", (x, y), fontsize=8)

plt.title("Embeddings visualized in 2D (Blue=Tech, Red=Food, Green=Nature)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.show()

print("\nNotice how similar topics cluster together.")

# **II/ Building a Vector Database**

Now that you understand how embeddings capture meaning as numbers, the next question is : **how do we search through thousands of embeddings quickly ?**

When you have 10 documents, you can compare them one by one. But with 10,000 or 1,000,000 documents, you need something smarter. That's where **vector databases** come in.

Now that we understand embeddings, we need a place to **store** and **search** them efficiently.

**Why not just use a Python list ?**  
You could store embeddings in a list and loop through all of them to find the most similar one. But this gets **very slow** when you have thousands or millions of documents. Imagine comparing your query against 1 million vectors one by one - that would take forever.

A **vector database** is a specialized database designed to :
- Store embeddings (lists of numbers) efficiently
- Find the most similar vectors **extremely fast**, even with millions of entries (using smart indexing algorithms)
- Handle metadata (like the source file, date, author...) alongside each vector

We will use **ChromaDB**, which is simple, works locally, and is perfect for learning.

**Important concept :** In Part I, we manually created embeddings using `SentenceTransformer`. ChromaDB can do this **automatically** for you ! When you add a text document, ChromaDB will create its embedding behind the scenes using its own built-in embedding model (which happens to be `all-MiniLM-L6-v2` - the same one we used in Part I !).

So Part I taught you **how embeddings work under the hood**. Now ChromaDB will handle the embedding step for us, so we can focus on building the search and retrieval pipeline.

**Other popular vector databases :** Pinecone, Weaviate, Qdrant, FAISS, Milvus

### ***1/ Create a ChromaDB collection***

A "collection" in ChromaDB is like a table in a regular database.  
It stores your documents and their embeddings together.

When you create a collection, ChromaDB is ready to :
1. Accept documents (plain text)
2. Automatically convert them to embeddings
3. Store both the text and its embedding
4. Later, search for similar documents when you ask a question

**Documentation :** https://docs.trychroma.com/

In [ ]:
import chromadb

# TODO: Create a ChromaDB client
chroma_client = ...

# TODO: Create a collection named "my_knowledge_base"
collection = ...

print(f"Collection '{collection.name}' created.")
print(f"Currently contains {collection.count()} documents")

### ***2/ Add documents to the database***

Let's add some documents about a fictional company.  
Later, we will ask questions and retrieve relevant information.

**Your task :** Add documents to the collection using the `add()` method.

**Hint :** ChromaDB requires each document to have a **unique string ID**.  
Check the documentation to see how the `add()` method works : https://docs.trychroma.com/docs/collections/add-data

In [ ]:
documents = [
    "TechCorp was founded in 2020 by Alice Johnson and Bob Smith in San Francisco.",
    "TechCorp specializes in artificial intelligence solutions for healthcare.",
    "The company has 150 employees and offices in San Francisco and London.",
    "TechCorp's main product is MedAI, a diagnostic assistant for doctors.",
    "In 2023, TechCorp raised $50 million in Series B funding from Sequoia Capital.",
    "The CEO of TechCorp is Alice Johnson, who previously worked at Google.",
    "TechCorp's revenue in 2023 was $25 million, a 150% increase from 2022.",
    "The company plans to expand to Asia in 2024, starting with Japan and Singapore.",
    "MedAI can analyze X-rays, MRIs, and CT scans with 95% accuracy.",
    "TechCorp won the Best AI Startup award at TechCrunch Disrupt 2023."
]

# TODO: Add documents to the collection with unique IDs
...

print(f"Added {collection.count()} documents to the collection.")

### ***3/ Search for relevant documents***

Now the magic happens. We can search for documents by **meaning**, not just keywords.  
The database will find documents that are semantically similar to our query.

**Your task :** Use the `query()` method to search for relevant documents.

In [ ]:
query = "Who founded the company and when ?"

# TODO: Query the collection and get 3 results
results = ...

print(f"Query: \"{query}\"\n")
print("Most relevant documents:")
for i, doc in enumerate(results['documents'][0]):
    print(f"  {i+1}. {doc}")

### ***4/ Experiment with different queries***

Try different questions and see how the system finds relevant documents.  
Notice how it understands meaning, not just exact word matches.

In [ ]:
test_queries = [
    "What does the company sell ?",
    "How much money did they raise ?",
    "Where are the offices located ?",
    "Tell me about the medical AI product"
]

for query in test_queries:
    # TODO: Query the collection with 2 results
    results = ...
    
    print(f"\nQuery: \"{query}\"")
    print("Results:")
    for doc in results['documents'][0]:
        print(f"   → {doc}")
    print("-" * 60)

# **III/ Building a RAG System**

In Part II, we built a system that can **find relevant documents** based on a question. That's great, but it only gives us raw text snippets - it doesn't actually **answer** the question.

Now we combine everything into a complete **RAG** (Retrieval Augmented Generation) system by adding an LLM to the pipeline.

**How RAG works - the 3 steps :**
1. **Retrieve** : The user asks a question. We search our vector database for the most relevant documents.
2. **Augment** : We take those documents and insert them into a prompt, along with the question. This gives the LLM the context it needs.
3. **Generate** : The LLM reads the context and generates an answer based **only on the provided documents**.

**Why is this powerful ?**
- The LLM has access to **your specific data** (company docs, internal knowledge...)
- Answers are **grounded** in real documents, which reduces hallucination
- You can **update** the knowledge base anytime without retraining the model
- You can trace **which documents** were used to generate each answer

### ***1/ Setup the LLM***

For RAG, we need an LLM that can read our documents and generate an answer. We will use **Ollama** to run a local LLM.

**Why Ollama and not HuggingFace like this morning ?**  
This morning, we used HuggingFace `transformers` to load GPT-2 because we needed to **modify** the model (fine-tuning). Here, we don't need to modify anything - we just need to **send a prompt and get a response**. Ollama makes this very simple :
- It handles downloading and running LLMs with a single command
- It provides a simple HTTP API (like a web server) that we can call from Python
- It supports powerful models like Llama 3.2 that are much better than GPT-2 for generating answers

Think of Ollama as a "model server" : it runs in the background and we send it questions via HTTP requests.

**Setup steps :**

1. Install Ollama from [ollama.com](https://ollama.com/)
2. Open a **separate terminal** and run :
```bash
ollama pull llama3.2:3b
```
3. Keep Ollama running in the background :
```bash
ollama serve
```

**Troubleshooting :**
- If you get a "connection refused" error later, it means Ollama is not running. Start it with `ollama serve` in a terminal.
- If `ollama pull` fails, check your internet connection.
- The model is ~2GB, so the first download may take a few minutes.

**Documentation :** https://github.com/ollama/ollama/blob/main/docs/api.md#generate-a-completion

In [ ]:
%pip install requests

import requests

LLM_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "llama3.2:3b"

# Test the connection to Ollama
try:
    test_response = requests.get("http://localhost:11434/api/tags", timeout=5)
    if test_response.status_code == 200:
        models = [m["name"] for m in test_response.json().get("models", [])]
        print(f"Ollama is running !")
        print(f"Available models: {models}")
        if not any(LLM_MODEL in m for m in models):
            print(f"\nWARNING: '{LLM_MODEL}' not found. Run: ollama pull {LLM_MODEL}")
        else:
            print(f"Model '{LLM_MODEL}' is ready.")
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot connect to Ollama !")
    print("Make sure Ollama is running: open a terminal and run 'ollama serve'")
    print("Then run: ollama pull llama3.2:3b")

### ***2/ Build the RAG pipeline***

Let's create a function that implements the 3 RAG steps :
1. **Retrieve** : Query ChromaDB to find relevant documents
2. **Augment** : Build a prompt that includes the retrieved documents as context
3. **Generate** : Send the prompt to Ollama and get the answer

**Your task :** Complete the RAG function below.

**Hint 1 - Building the context :** You need to combine the retrieved documents into a single block of text that the LLM can read.

**Hint 2 - Structuring the prompt :** A good RAG prompt should :
- Clearly separate the context (retrieved documents) from the question
- Instruct the LLM to answer **only** based on the provided context
- Handle the case where the answer is not in the context

Think about what instructions you would give to a human if you handed them a stack of documents and asked them a question.

**Hint 3 - Calling Ollama :** Ollama exposes a REST API. You need to send a **POST** request with a JSON body containing the model name and your prompt. Make sure to set `"stream": False` to get the full response at once (otherwise Ollama streams the response token by token in a different format).

**Documentation :** https://github.com/ollama/ollama/blob/main/docs/api.md#generate-a-completion

In [ ]:
def ask_with_rag(question: str, n_results: int = 3) -> tuple[str, list]:
    """
    RAG pipeline: Retrieve relevant docs and generate an answer.
    
    Args:
        question: The user's question
        n_results: Number of documents to retrieve
    
    Returns:
        Tuple of (answer, source documents)
    """
    
    # Step 1 - RETRIEVE: Query the collection to find relevant documents
    results = ...
    
    # Step 2 - AUGMENT: Build the context string from retrieved documents
    context = ...
    
    # Step 3 - AUGMENT: Create the prompt with context + question
    prompt = ...
    
    # Step 4 - GENERATE: Call Ollama API and extract the response
    response = ...
    answer = ...
    
    return answer, results['documents'][0]

print("RAG function created.")

### ***3/ Test your RAG system***

Now let's test our RAG system with various questions.

In [ ]:
test_questions = [
    "Who is the CEO of TechCorp ?",
    "What is MedAI and what can it do ?",
    "How much funding did the company raise ?",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}\n")
    
    try:
        answer, sources = ask_with_rag(question)
        print(f"Answer: {answer}")
        print(f"\nSources used:")
        for source in sources:
            print(f"   - {source}")
    except requests.exceptions.ConnectionError:
        print("ERROR: Cannot connect to Ollama.")
        print("Open a terminal and run: ollama serve")
    except KeyError as e:
        print(f"ERROR: Unexpected response format from Ollama: {e}")
        print("Make sure the model is downloaded: ollama pull llama3.2:3b")
    except Exception as e:
        print(f"ERROR: {e}")
        print("Check that Ollama is running and the model is available.")

### ***4/ Compare: With RAG vs Without RAG***

Let's see the difference between asking the LLM directly vs using RAG.  
This shows why RAG is so powerful for domain-specific questions.

In [ ]:
def ask_without_rag(question: str) -> str:
    """Ask the LLM directly without any context."""
    # TODO: Send the question directly to Ollama, without any context documents
    # Hint: same pattern as before, but the prompt is just the question itself
    ...

question = "Who is the CEO of TechCorp and what is their background ?"

print(f"Question: {question}\n")
print("=" * 60)

try:
    print("\nWITHOUT RAG (LLM has no context about TechCorp):")
    print(ask_without_rag(question))
    
    print("\n" + "=" * 60)
    print("\nWITH RAG (LLM receives relevant documents as context):")
    answer, _ = ask_with_rag(question)
    print(answer)
except requests.exceptions.ConnectionError:
    print("ERROR: Cannot connect to Ollama. Run 'ollama serve' in a terminal.")
except Exception as e:
    print(f"ERROR: {e}")

# **IV/ RAG on Real Documents : Chunking & Multi-File Pipeline**

### ***1/ Understanding chunking***

In Parts II and III, we worked with short, single-sentence documents. That made things easy.  
But in real applications, your knowledge base is made of **long documents** : PDFs, reports, articles, internal docs...

You can't just embed an entire 10-page document as a single vector. Why ?
- Embeddings work best on **short texts** (a few sentences). A single embedding for a whole document would lose the details.
- When you retrieve a long document, most of it is **irrelevant** to the question. You'd waste the LLM's context window.
- LLMs have **context limits** - you can't feed them an entire book.

The solution is **chunking** : splitting long documents into smaller, meaningful pieces.

**How chunking works :**
- We define a **maximum chunk size** (e.g. 500 characters).
- We walk through the text and cut at approximately every 500 characters.
- But we don't cut in the middle of a sentence ! We look for the **last sentence boundary** (period, exclamation mark...) before the limit, so each chunk contains **complete sentences**.
- We also add an **overlap** between chunks (e.g. 100 characters). This means the end of one chunk is repeated at the start of the next one. This prevents losing context at the boundaries - if an important fact spans two chunks, the overlap ensures it appears fully in at least one of them.

**Example with chunk_size=500, overlap=100 :**
```
Document: "Sentence A. Sentence B. Sentence C. Sentence D. Sentence E. ..."

Chunk 1: "Sentence A. Sentence B. Sentence C."        (480 chars, cut at last period before 500)
Chunk 2: "Sentence C. Sentence D. Sentence E."          (starts 100 chars before the end of chunk 1)
```

In the `documents/` folder, you will find **5 text files** about TechCorp.  
Your task is to implement the chunking function, load the files, chunk them, and build a complete RAG system over real documents.

**Hint :** Python's `str` methods like `rfind()` can help you find sentence boundaries within a range of text.

In [ ]:
import os

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list:
    """
    Split text into overlapping chunks, cutting at sentence boundaries.
    
    Args:
        text: The full text to chunk
        chunk_size: Maximum size of each chunk (in characters)
        overlap: Number of characters to overlap between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        # TODO: Find where this chunk should end.
        #       - Don't exceed chunk_size characters
        #       - Try to cut at a sentence boundary (not in the middle of a sentence)
        end = ...
        
        # TODO: Extract the chunk, add it to the list, and advance start
        #       - Don't forget the overlap when moving start forward
        #       - Make sure you can't get stuck in an infinite loop
        ...
    
    return chunks


# --- Load all .txt files from the documents/ folder ---
documents_dir = "documents"
all_chunks = []
chunk_sources = []

for filename in sorted(os.listdir(documents_dir)):
    if not filename.endswith(".txt"):
        continue
    
    filepath = os.path.join(documents_dir, filename)
    with open(filepath, "r") as f:
        content = f.read()
    
    # TODO: Chunk the file content using the chunk_text function
    file_chunks = ...
    
    for chunk in file_chunks:
        all_chunks.append(chunk)
        chunk_sources.append(filename)
    
    print(f"Loaded '{filename}' -> {len(file_chunks)} chunks")

print(f"\nTotal: {len(all_chunks)} chunks from {len(set(chunk_sources))} files")
print(f"\nExample chunk (chunk #1):")
print(f"  Source: {chunk_sources[0]}")
print(f"  Length: {len(all_chunks[0])} chars")
print(f"  Content: \"{all_chunks[0][:150]}...\"")

### ***2/ Store chunks in a vector database***

Now that we have chunks from multiple files, let's store them in a **new ChromaDB collection** and build a full RAG system over real documents.

**Your task :** Add all chunks to a new collection, keeping track of which file each chunk came from (using **metadata**).

In [ ]:
# TODO: Create a new ChromaDB collection named "techcorp_docs"
docs_collection = ...

# TODO: Add all chunks to the collection
# Each chunk needs: a unique ID, the chunk text as document, and metadata with the source filename
# Hint: metadata is a list of dicts, e.g. [{"source": "file1.txt"}, {"source": "file2.txt"}, ...]
...

print(f"Stored {docs_collection.count()} chunks in the 'techcorp_docs' collection.")

### ***3/ RAG over real documents***

Let's test our complete pipeline : **chunked documents + vector DB + LLM**.  
The questions below require information spread across different files. Only a RAG system with proper chunking can answer them accurately.

In [ ]:
def ask_docs(question: str, n_results: int = 5) -> tuple[str, list]:
    """RAG pipeline over the chunked documents collection."""
    
    # TODO: Query the docs_collection for relevant chunks
    results = ...
    
    # TODO: Build context from retrieved chunks
    context = ...
    
    # TODO: Create a prompt (same structure as ask_with_rag - context + question + instruction)
    prompt = ...
    
    response = requests.post(LLM_URL, json={
        "model": LLM_MODEL,
        "prompt": prompt,
        "stream": False
    })
    answer = response.json()["response"]
    
    return answer, results['documents'][0], results['metadatas'][0]


test_questions = [
    "What is TechCorp's revenue growth from 2022 to 2023 ?",
    "Which hospitals are partners of TechCorp ?",
    "What is PathAI and when will it launch ?",
    "How does MedAI integrate into hospital workflows ?",
    "What is TechCorp's expansion plan for Asia ?",
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}\n")
    
    try:
        answer, sources, metadatas = ask_docs(question)
        print(f"Answer: {answer}")
        print(f"\nSources:")
        for source, meta in zip(sources, metadatas):
            print(f"   [{meta['source']}] {source[:80]}...")
    except requests.exceptions.ConnectionError:
        print("ERROR: Cannot connect to Ollama. Run 'ollama serve' in a terminal.")
    except Exception as e:
        print(f"ERROR: {e}")

# **Conclusion**

---

**Congratulations !** You have completed this afternoon's session on RAG.

**What you learned today :**

- **Embeddings** : Transform text into vectors that capture meaning  
- **Cosine similarity** : Measure how close two texts are in meaning  
- **Vector Databases** : Store and search embeddings efficiently (ChromaDB)  
- **RAG Pipeline** : Retrieve relevant documents + Generate answers with an LLM  
- **Chunking** : Split large documents into smaller pieces for better retrieval  

**Key takeaways :**
- RAG lets you give LLMs access to **your specific data** without retraining
- Embeddings enable **semantic search** (by meaning, not just keywords)
- The quality of your RAG system depends on **how you chunk** your documents and **how you write your prompt**

**What's next ? Ideas to explore :**
- Build a RAG system with your own documents (PDFs, web pages)
- Try different embedding models and compare results
- Experiment with different chunk sizes and overlaps to see how it affects quality
---

**Combining this morning and this afternoon :**  
This morning you learned **fine-tuning** (adapting model weights to learn new behavior).  
This afternoon you learned **RAG** (giving the model external knowledge at query time).  

In practice, production AI systems often use **both** :
- **Fine-tune** for style, format, or domain-specific language (how the model speaks)
- **RAG** for factual, up-to-date information (what the model knows)